<a href="https://colab.research.google.com/github/achyutmorang/waymax-simulation-experiments/blob/main/experiments/risk-uq-suite/notebooks/10_decision_audit/decision_audit_artifact_builder_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Open In Colab\n\n[![Run in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/waymax-simulation-experiments/blob/main/experiments/risk-uq-suite/notebooks/10_decision_audit/decision_audit_artifact_builder_colab.ipynb)


# Risk/UQ Track: Decision-Audit Artifact Builder (Closed-Loop Producer)

## Objective
Run the exact closed-loop miscalibration probe pipeline that the decision-audit notebooks depend on, and persist all required artifacts to Google Drive.

## Inputs
- Experiment config from `configs/experiments/risk-uq-suite.json`
- Waymax/LatentDriver runtime bootstrap in Colab

## Outputs (for downstream notebooks)
- `*_miscalibration_probe_predictions.parquet`
- `*_miscalibration_probe_summary.csv`
- `*_miscalibration_probe_threshold_diagnostics.csv`
- `*_miscalibration_probe_reliability_bins.csv`
- `*_miscalibration_probe_class_balance.csv`
- Contract manifest files mirrored to persistent storage

## Downstream Consumers
1. `cross_signal_decision_audit_colab.ipynb`
2. `oracle_bottleneck_colab.ipynb`
3. `planner_practice_method_benchmark_colab.ipynb`


## Execution Design
- This notebook is the producer stage for decision-audit artifacts.
- It can run in **fresh mode** (default here) so you generate a new run prefix and avoid accidental reuse.
- Artifacts are written under `PERSIST_ROOT` in Google Drive and later discovered by analysis notebooks using `analysis_run_prefix` or auto-discovery.

## Reliability Requirements
1. Mount Drive and verify write access before running simulation.
2. Run smoke gates before main execution.
3. Export notebook contract manifests at start and completion.
4. Keep `RUN_ENABLED=True` unless intentionally dry-running setup only.
- This notebook now persists the selected run tag to Drive and auto-resumes that run after Colab restart.



## Step 1 - Deterministic Bootstrap Constants


In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/achyutmorang/waymax-simulation-experiments.git'
REPO_DIR = '/content/waymax-simulation-experiments'
REPO_BRANCH = 'main'
EXPERIMENT_SLUG = 'risk-uq-suite'
EXPERIMENT_CONFIG_PATH = f'{REPO_DIR}/configs/experiments/{EXPERIMENT_SLUG}.json'
REQUIRED_DRIVE_FOLDER = '/content/drive/MyDrive/waymax_experiments'

runtime_cfg_overrides = {
    'verify_drive_access_every_run': False,
    'force_reinstall': False,
    'auto_restart_after_setup': True,
    'strict_lockfile_check': True,
    'setup_cache_enabled': True,
    'revalidate_core_imports_on_cache_hit': True,
    'setup_cache_path': '/content/.closedloop_setup_cache.json',
    'force_module_hot_reload': True,
}

content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))

NOTEBOOK_NAME = 'decision_audit_artifact_builder_colab'


## Step 2 - Storage Setup


In [2]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    print('[storage] google.colab drive mount unavailable:', exc)

drive_root = Path('/content/drive/MyDrive')
if not drive_root.exists():
    raise RuntimeError('Drive root not found at /content/drive/MyDrive. Ensure Google Drive is mounted.')

required_root = Path(REQUIRED_DRIVE_FOLDER)
required_root.mkdir(parents=True, exist_ok=True)
probe = required_root / '.risk_uq_storage_probe'
probe.write_text('ok')
probe.unlink(missing_ok=True)
print('[storage] ready:', required_root)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[storage] ready: /content/drive/MyDrive/waymax_experiments


## Step 3 - Repo Sync + Runtime Bootstrap


In [3]:
if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

for path in (REPO_DIR, f'{REPO_DIR}/src'):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import ColabRuntimeConfig, bootstrap_colab_runtime_with_config

runtime_cfg = ColabRuntimeConfig(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    repo_branch=REPO_BRANCH,
    required_drive_folder=REQUIRED_DRIVE_FOLDER,
    **runtime_cfg_overrides,
)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)

globals()['_RISK_UQ_REPO_REV'] = str(bootstrap.repo_sync.repo_rev)
setup_result = bootstrap.setup.result

print('Working directory:', os.getcwd())
print('Repo commit:', bootstrap.repo_sync.repo_rev)
print('Drive mounted:', bootstrap.drive_status.mounted)
print('[setup] result:', setup_result)
if bool(setup_result.get('restart_required', False)) and (not bool(runtime_cfg.auto_restart_after_setup)):
    print('[setup] restart_required=True; restart runtime before continuing.')


[repo] existing checkout: /content/waymax-simulation-experiments
[drive] /content/drive already mounted
[drive] Verified read/write access: /content/drive/MyDrive/waymax_experiments
[setup] Starting deterministic environment bootstrap
[setup] Lockfile version check: no exact-pin mismatches detected.
[setup] $ /usr/bin/python3 -c import jax, waymax, numpy as np, pandas as pd, scipy, sklearn; from numpy._core.umath import _center, _expandtabs; print('ok', np.__version__, pd.__version__, scipy.__version__, sklearn.__version__, jax.__version__)
[setup] Core runtime already healthy; skipping dependency install (ok 2.2.6 2.2.3 1.14.1 1.6.1 0.7.2).
[setup] LatentDriver repo already exists: /content/LatentDriver
[patch] gpt2_model block already patched or changed
[patch] sort_vert fallback patch applied
[ckpt] already present: /content/checkpoints/lantentdriver_t2_J3.ckpt
[setup] $ /usr/bin/python3 -c import numpy as np; from numpy._core.umath import _center, _expandtabs; print(np.__version__,

## Step 4 - Configuration + Run Context (Fresh Producer Mode)


In [4]:
from pathlib import Path

from src.workflows import (
    initialize_risk_uq_run_context,
    load_experiment_config,
    report_risk_uq_run_context,
)

EXPERIMENT_CFG = load_experiment_config(
    slug=EXPERIMENT_SLUG,
    repo_root=REPO_DIR,
    default_on_missing=False,
)
run_cfg = dict(EXPERIMENT_CFG.get('run', {}))

# Mandatory contract fields
RUN_NAME = str(run_cfg.get('run_name', '')).strip()
RUN_PREFIX = str(run_cfg.get('run_prefix', 'risk_uq')).strip() or 'risk_uq'
PERSIST_ROOT = str(run_cfg.get('persist_root', '/content/drive/MyDrive/waymax_experiments/risk_uq_runs/v2')).strip()
N_SHARDS = int(max(1, int(run_cfg.get('n_shards', 1))))
SHARD_ID = run_cfg.get('shard_id', 'auto')
RESUME_FROM_EXISTING = bool(run_cfg.get('resume_from_existing', True))
RUN_ENABLED = bool(run_cfg.get('run_enabled', True))

RUN_TAG_PREFIX = str(run_cfg.get('run_tag_prefix', RUN_PREFIX)).strip() or RUN_PREFIX
RUN_MODE_CFG = str(run_cfg.get('run_mode', 'auto')).strip().lower() or 'auto'
RUN_MODE = RUN_MODE_CFG if RESUME_FROM_EXISTING else 'fresh'
RUN_TAG = RUN_NAME

# Producer behavior:
# - First launch: force fresh run (default)
# - Later launches (after runtime restarts): auto-resume same run from sentinel file in Drive
FORCE_FRESH_PROBE_BUILD = bool(run_cfg.get('decision_audit_force_fresh_probe', True))
AUTO_RESUME_AFTER_RESTART = bool(run_cfg.get('decision_audit_auto_resume_after_restart', True))

SENTINEL_DIR = Path(PERSIST_ROOT) / RUN_PREFIX
SENTINEL_PATH = SENTINEL_DIR / '_decision_audit_artifact_builder_last_run.txt'
SAVED_RUN_TAG = ''
if AUTO_RESUME_AFTER_RESTART and SENTINEL_PATH.exists():
    try:
        SAVED_RUN_TAG = SENTINEL_PATH.read_text(encoding='utf-8').strip()
    except Exception:
        SAVED_RUN_TAG = ''

if AUTO_RESUME_AFTER_RESTART and SAVED_RUN_TAG:
    RUN_NAME = SAVED_RUN_TAG
    RUN_TAG = SAVED_RUN_TAG
    RESUME_FROM_EXISTING = True
    RUN_MODE = 'resume'
    FORCE_FRESH_PROBE_BUILD = False
elif FORCE_FRESH_PROBE_BUILD:
    RESUME_FROM_EXISTING = False
    RUN_MODE = 'fresh'
    RUN_NAME = ''
    RUN_TAG = ''

run_ctx = initialize_risk_uq_run_context(
    run_tag=RUN_TAG,
    run_tag_prefix=RUN_TAG_PREFIX,
    persist_root=PERSIST_ROOT,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    resume_mode=RUN_MODE,
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    planner_backend='latentdriver',
)

cfg = run_ctx.cfg
search_cfg = {}
RUN_TAG = str(run_ctx.run_tag)
RUN_NAME = str(RUN_NAME or RUN_TAG)
SHARD_ID = int(run_ctx.shard_id)
run_prefix = run_ctx.run_prefix

# Persist selected run tag immediately so a runtime restart can resume this exact run.
if AUTO_RESUME_AFTER_RESTART:
    try:
        SENTINEL_DIR.mkdir(parents=True, exist_ok=True)
        SENTINEL_PATH.write_text(str(RUN_TAG), encoding='utf-8')
    except Exception as exc:
        print('[resume sentinel] warning:', exc)

print('EXPERIMENT_CONFIG_PATH =', EXPERIMENT_CONFIG_PATH)
print('run_prefix (flat artifacts) =', run_prefix)
print('RUN_PREFIX/RUN_NAME (contract dir) =', RUN_PREFIX, RUN_NAME)
print('RUN_ENABLED =', RUN_ENABLED, ' RESUME_FROM_EXISTING =', RESUME_FROM_EXISTING)
print('FORCE_FRESH_PROBE_BUILD =', FORCE_FRESH_PROBE_BUILD)
print('AUTO_RESUME_AFTER_RESTART =', AUTO_RESUME_AFTER_RESTART)
print('SENTINEL_PATH =', SENTINEL_PATH)
report_risk_uq_run_context(run_ctx, display_fn=display)

# Probe knobs
cfg.uq_eval_probability_bins = 15
cfg.risk_control_fail_budget = 0.20
cfg.uq_shift_suites = ('nominal_clean', 'hist_prim_shift', 'fut_prim_shift', 'hist_rmv_shift', 'high_interaction_holdout')
cfg


[config auto-fix] latentdriver_context_len increased from 2 to 10 to match history_steps.
[ckpt] using configured checkpoint: /content/checkpoints/lantentdriver_t2_J3.ckpt
[ckpt] final cfg.latentdriver_ckpt_path = /content/checkpoints/lantentdriver_t2_J3.ckpt
EXPERIMENT_CONFIG_PATH = /content/waymax-simulation-experiments/configs/experiments/risk-uq-suite.json
run_prefix (flat artifacts) = /content/drive/MyDrive/waymax_experiments/risk_uq_runs/v2/risk_uq_v2_20260301_121046
RUN_PREFIX/RUN_NAME (contract dir) = risk_uq_v2 risk_uq_v2_20260301_121046
RUN_ENABLED = True  RESUME_FROM_EXISTING = False
FORCE_FRESH_PROBE_BUILD = True
AUTO_RESUME_AFTER_RESTART = True
SENTINEL_PATH = /content/drive/MyDrive/waymax_experiments/risk_uq_runs/v2/risk_uq_v2/_decision_audit_artifact_builder_last_run.txt


,field,value
0,run_tag,risk_uq_v2_20260301_121046
1,run_prefix,/content/drive/MyDrive/waymax_experiments/risk...
2,run_mode,fresh
3,planner_backend,latentdriver
4,persist_root,/content/drive/MyDrive/waymax_experiments/risk...
5,n_shards,1
6,shard_id,0
7,resume_from_existing,0
8,latentdriver_ckpt_path,/content/checkpoints/lantentdriver_t2_J3.ckpt


,path,name,size_mb,mtime_utc,root,root_rank,name_score,score
0,/content/checkpoints/lantentdriver_t2_J3.ckpt,lantentdriver_t2_J3.ckpt,116.742193,2026-03-01T12:08:43Z,/content/checkpoints,0,100,100100.011674


ClosedLoopConfig(global_seed=17, n_total_scenarios=2400, n_eval_scenarios=200, strict_min_eval=200, train_fraction=0.75, n_agents=8, history_steps=10, future_steps=15, waymax_path='gs://waymo_open_dataset_motion_v_1_1_0/uncompressed/tf_example/training/training_tfexample.tfrecord@1000', waymax_max_rg_points=20000, waymax_batch_dims=(), collision_distance=1.5, risk_w_dist=1.0, risk_w_ttc=0.5, risk_w_sks=0.01, ttc_fail_seconds=2.0, no_hazard_ttc_seconds=3.0, no_hazard_dist_m=8.0, hard_brake_mps2=6.0, hard_jerk_mps3=8.0, enable_intervention_proxy=True, planner_kind='latentdriver', planner_name='latentdriver_waypoint_sdc', planner_surprise_name='latent_belief_kl', surprise_belief_source_mode='auto', predictive_kl_estimator='mixture_mc', predictive_kl_mc_samples=192, predictive_kl_mc_seed=12345, predictive_kl_eps=1e-06, predictive_kl_symmetric=True, predictive_kl_skip_fallback_steps=True, surprise_counterfactual_floor_weight=0.02, surprise_counterfactual_response_weight=0.35, surprise_count

## Step 5 - Closed-Loop Context + Smoke Gates


In [6]:
import numpy as np
import pandas as pd
from src.workflows import (
    build_risk_uq_simulation_context,
    load_existing_risk_dataset_artifact,
    run_risk_uq_smoke_gates,
)

existing_dataset_df = load_existing_risk_dataset_artifact(cfg.run_prefix)
if RUN_MODE == 'fresh':
    existing_dataset_df = pd.DataFrame()

needs_simulation_context = bool(existing_dataset_df.empty)
runner = None
eval_idx = None

print('existing_dataset_rows =', len(existing_dataset_df))
print('needs_simulation_context =', needs_simulation_context)

if needs_simulation_context:
    sim_bundle = build_risk_uq_simulation_context(cfg=cfg, n_shards=N_SHARDS, shard_id=SHARD_ID)
    runner = sim_bundle.runner
    eval_idx = sim_bundle.eval_idx

    gate_bundle = run_risk_uq_smoke_gates(
        runner=runner,
        cfg=cfg,
        eval_idx=eval_idx,
        probe_shift_suite='nominal_clean',
    )
    risk_probe_pass = bool(gate_bundle.get('risk_probe_pass', False))
    preflight_pass = bool(gate_bundle.get('preflight_pass', False))
    overall_pass = bool(gate_bundle.get('overall_pass', False))

    display(gate_bundle.get('risk_probe_summary_df', pd.DataFrame()))
    if isinstance(gate_bundle.get('preflight_df', None), pd.DataFrame):
        display(gate_bundle.get('preflight_df', pd.DataFrame()))
else:
    gate_bundle = {'failure_reasons': []}
    required_cols = [
        'scenario_id',
        'dist_top1_weight',
        'dist_entropy',
        'dist_num_components',
        'failure_proxy_h15',
    ]
    required_ok = all(col in existing_dataset_df.columns for col in required_cols)

    finite_ok = False
    if required_ok and len(existing_dataset_df) > 0:
        sample = existing_dataset_df.loc[:, ['dist_top1_weight', 'dist_entropy', 'dist_num_components', 'failure_proxy_h15']].head(4096)
        finite_ok = bool(np.isfinite(sample.to_numpy(dtype=float)).all())

    risk_probe_pass = bool((len(existing_dataset_df) > 0) and required_ok and finite_ok)
    preflight_pass = bool(risk_probe_pass)
    overall_pass = bool(risk_probe_pass and preflight_pass)

print('risk_probe_pass =', risk_probe_pass)
print('preflight_pass =', preflight_pass)
print('overall_pass =', overall_pass)

if not overall_pass:
    raise RuntimeError(f"Miscalibration probe gates failed: {gate_bundle.get('failure_reasons', [])}")


existing_dataset_rows = 0
needs_simulation_context = True
[auth] GCS access already available for WOMD bucket.


Loading WOMD scenarios:   0%|          | 0/2400 [00:00<?, ?it/s]

Building feature table:   0%|          | 0/2400 [00:00<?, ?it/s]

Scoring open-loop reference_openloop:   0%|          | 0/1800 [00:00<?, ?it/s]

Scoring open-loop base_eval_openloop:   0%|          | 0/200 [00:00<?, ?it/s]

...

,check,pass,detail
0,planner_bundle_constructed,1,LatentDriverPredictiveKL
1,risk_probe_rows_nonempty,1,rows=8
2,risk_probe_numeric_finite,1,all numeric columns finite
3,risk_probe_required_columns,1,dist_entropy/progress/labels present
4,preflight_all_checks_pass,1,risk-uq smoke gates only


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,check,pass,detail
0,planner_bundle_constructed,1,LatentDriverPredictiveKL
1,risk_probe_rows_nonempty,1,rows=8
2,risk_probe_numeric_finite,1,all numeric columns finite
3,risk_probe_required_columns,1,dist_entropy/progress/labels present
4,preflight_all_checks_pass,1,risk-uq smoke gates only


risk_probe_pass = True
preflight_pass = True
overall_pass = True


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Step 6 - Manifest + Contract Layout (Gates Passed)


In [7]:
from src.workflows import write_contract_storage_mirror, write_notebook_contract_manifest

manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name=NOTEBOOK_NAME,
    stage='decision_audit_artifact_build_started',
    repo_dir=REPO_DIR,
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
    quick_probe_pass=bool(risk_probe_pass),
    preflight_pass=bool(preflight_pass),
)
contract_paths = write_contract_storage_mirror(
    persist_root=PERSIST_ROOT,
    run_prefix=RUN_PREFIX,
    run_name=RUN_NAME,
    run_prefix_path=cfg.run_prefix,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    stage='decision_audit_artifact_build_started',
    git_commit=str(globals().get('_RISK_UQ_REPO_REV', '')),
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    run_enabled=bool(RUN_ENABLED),
)
print('manifest_path =', manifest_path)
print('contract_run_dir =', contract_paths['contract_run_dir'])


manifest_path = /content/drive/MyDrive/waymax_experiments/risk_uq_runs/v2/risk_uq_v2_20260301_121046_notebook_contract_manifest.json
contract_run_dir = /content/drive/MyDrive/waymax_experiments/risk_uq_runs/v2/risk_uq_v2_risk_uq_v2_20260301_121046


## Step 7 - Main Execution (Generate Probe Artifacts)


In [8]:
from src.workflows import run_miscalibration_probe_flow

# Run exact probe flow used by decision-audit notebooks.
probe_bundle = None
if not bool(RUN_ENABLED):
    print('[main] skipped: RUN_ENABLED=False')
else:
    probe_bundle = run_miscalibration_probe_flow(
        cfg=cfg,
        dataset_df=existing_dataset_df if not existing_dataset_df.empty else None,
        runner=runner,
        eval_idx=eval_idx,
        run_prefix=cfg.run_prefix,
        resume_mode=RUN_MODE,
    )
    print('loaded_from_existing =', probe_bundle.loaded_from_existing)
    display(probe_bundle.benchmark_bundle.summary_df)


[planner notice] requested=latentdriver_waypoint_sdc, used=LatentDriverPredictiveKL


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[planner notice] requested=latentdriver_waypoint_sdc, used=LatentDriverPredictiveKL
[planner notice] requested=latentdriver_waypoint_sdc, used=LatentDriverPredictiveKL
[planner notice] requested=latentdriver_waypoint_sdc, used=LatentDriverPredictiveKL
[planner notice] requested=latentdriver_waypoint_sdc, used=LatentDriverPredictiveKL
[planner notice] requested=latentdriver_waypoint_sdc, used=LatentDriverPredictiveKL
[planner notice] requested=latentdriver_waypoint_sdc, used=LatentDriverPredictiveKL
[planner notice] requested=latentdriver_waypoint_sdc, used=LatentDriverPredictiveKL
[planner notice] requested=latentdriver_waypoint_sdc, used=LatentDriverPredictiveKL
[planner notice] requested=latentdriver_waypoint_sdc, used=LatentDriverPredictiveKL
[planner notice] requested=latentdriver_waypoint_sdc, used=LatentDriverPredictiveKL
[planner notice] requested=latentdriver_waypoint_sdc, used=LatentDriverPredictiveKL
[planner notice] requested=latentdriver_waypoint_sdc, used=LatentDriverPredi

/content/waymax-simulation-experiments/src/risk_model/benchmark.py:91: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(precision, recall))
/content/waymax-simulation-experiments/src/risk_model/benchmark.py:91: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(precision, recall))
/content/waymax-simulation-experiments/src/risk_model/benchmark.py:91: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(precision, recall))
/content/waymax-simulation-experiments/src/risk_model/benchmark.py:91: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(precision, recal

loaded_from_existing = False


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,label,variant,n_rows,positive_rate,ece,adaptive_ece,brier,nll,auroc,auprc
0,failure_proxy_h15,planner_combo_platt,520,0.1125,0.119715,0.138315,0.096619,0.331584,0.736559,0.354548
1,failure_proxy_h15,planner_combo_raw,520,0.1125,0.770833,0.770833,0.694028,1.920691,0.736559,0.354548
2,failure_proxy_h15,planner_entropy_platt,520,0.1125,0.112500,0.176538,0.100518,0.354700,0.500000,0.481109
3,failure_proxy_h15,planner_entropy_raw,520,0.1125,0.887499,0.887499,0.887498,12.261266,0.670251,0.360157
4,failure_proxy_h15,planner_top1_platt,520,0.1125,0.119715,0.143593,0.096619,0.331584,0.743728,0.353425
5,failure_proxy_h15,planner_top1_raw,520,0.1125,0.720833,0.720833,0.619444,1.610697,0.743728,0.353425


## Step 8 - Evaluation/Diagnostics


In [9]:
if probe_bundle is None:
    print('[report] no run output because RUN_ENABLED=False')
else:
    display(probe_bundle.benchmark_bundle.per_shift_df.head(30))
    display(probe_bundle.benchmark_bundle.reliability_df.head(30))
    display(probe_bundle.threshold_df.head(30))
    if hasattr(probe_bundle, 'proxy_calibration_df'):
        display(probe_bundle.proxy_calibration_df.head(20))
    if hasattr(probe_bundle, 'leakage_df'):
        display(probe_bundle.leakage_df.head(20))
    if hasattr(probe_bundle, 'shift_profile_df'):
        display(probe_bundle.shift_profile_df.head(30))
    if hasattr(probe_bundle, 'class_balance_df'):
        display(probe_bundle.class_balance_df.head(30))

    key = probe_bundle.threshold_df.copy()
    if (not key.empty) and ('variant' in key.columns):
        cols = ['shift_suite', 'variant', 'empirical_failure_given_accepted', 'threshold', 'budget_violated']
        cols = [c for c in cols if c in key.columns]
        print('[diagnostic] budget check by shift/variant')
        display(key.loc[:, cols].head(30))


,label,shift_suite,variant,n_rows,positive_rate,ece,adaptive_ece,brier,nll,auroc,auprc
0,failure_proxy_h15,high_interaction_holdout,planner_top1_raw,320,0.225,0.608333,0.608333,0.544444,1.429636,0.743728,0.353425
1,failure_proxy_h15,nominal_clean,planner_top1_raw,200,0.000,0.833333,0.833333,0.694444,1.791759,NaN,NaN
2,failure_proxy_h15,high_interaction_holdout,planner_top1_platt,320,0.225,0.095336,0.143092,0.165950,0.503212,0.743728,0.353425
3,failure_proxy_h15,nominal_clean,planner_top1_platt,200,0.000,0.144094,0.144094,0.027288,0.159955,NaN,NaN
4,failure_proxy_h15,high_interaction_holdout,planner_entropy_raw,320,0.225,0.774999,0.774999,0.774998,10.707021,0.670251,0.360157
5,failure_proxy_h15,nominal_clean,planner_entropy_raw,200,0.000,0.999999,0.999999,0.999998,13.815511,NaN,NaN
6,failure_proxy_h15,high_interaction_holdout,planner_entropy_platt,320,0.225,0.086538,0.214615,0.181864,0.560364,0.500000,0.481109
7,failure_proxy_h15,nominal_clean,planner_entropy_platt,200,0.000,0.138462,0.138462,0.019172,0.149036,NaN,NaN
8,failure_proxy_h15,high_interaction_holdout,planner_combo_raw,320,0.225,0.658333,0.658333,0.607778,1.692948,0.736559,0.354548
9,failure_proxy_h15,nominal_clean,planner_combo_raw,200,0.000,0.883333,0.883333,0.780278,2.148434,NaN,NaN


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,bin_id,bin_lo,bin_hi,count,mean_prob,event_rate,label,variant,shift_suite
0,0,0.000000,0.066667,0,NaN,NaN,failure_proxy_h15,planner_top1_raw,high_interaction_holdout
1,1,0.066667,0.133333,0,NaN,NaN,failure_proxy_h15,planner_top1_raw,high_interaction_holdout
2,2,0.133333,0.200000,0,NaN,NaN,failure_proxy_h15,planner_top1_raw,high_interaction_holdout
3,3,0.200000,0.266667,0,NaN,NaN,failure_proxy_h15,planner_top1_raw,high_interaction_holdout
4,4,0.266667,0.333333,0,NaN,NaN,failure_proxy_h15,planner_top1_raw,high_interaction_holdout
5,5,0.333333,0.400000,0,NaN,NaN,failure_proxy_h15,planner_top1_raw,high_interaction_holdout
6,6,0.400000,0.466667,0,NaN,NaN,failure_proxy_h15,planner_top1_raw,high_interaction_holdout
7,7,0.466667,0.533333,0,NaN,NaN,failure_proxy_h15,planner_top1_raw,high_interaction_holdout
8,8,0.533333,0.600000,0,NaN,NaN,failure_proxy_h15,planner_top1_raw,high_interaction_holdout
9,9,0.600000,0.666667,0,NaN,NaN,failure_proxy_h15,planner_top1_raw,high_interaction_holdout


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,shift_suite,variant,label,threshold,n_rows,accepted_count,accepted_rate,mean_predicted_risk,empirical_failure_rate,risk_gap_empirical_minus_predicted,empirical_failure_given_accepted,false_safe_rate,safe_rejected_rate,budget_violated
0,high_interaction_holdout,planner_top1_raw,failure_proxy_h15,0.2,320,0,0.00,0.833333,0.225,-0.608333,NaN,0.000,0.775,0
1,high_interaction_holdout,planner_top1_platt,failure_proxy_h15,0.2,320,208,0.65,0.134942,0.225,0.090058,0.115385,0.075,0.200,0
2,high_interaction_holdout,planner_entropy_raw,failure_proxy_h15,0.2,320,0,0.00,0.999999,0.225,-0.774999,NaN,0.000,0.775,0
3,high_interaction_holdout,planner_entropy_platt,failure_proxy_h15,0.2,320,320,1.00,0.138462,0.225,0.086538,0.225000,0.225,0.000,1
4,high_interaction_holdout,planner_combo_raw,failure_proxy_h15,0.2,320,0,0.00,0.883333,0.225,-0.658333,NaN,0.000,0.775,0
5,high_interaction_holdout,planner_combo_platt,failure_proxy_h15,0.2,320,208,0.65,0.134942,0.225,0.090058,0.115385,0.075,0.200,0
6,nominal_clean,planner_top1_raw,failure_proxy_h15,0.2,200,0,0.00,0.833333,0.000,-0.833333,NaN,0.000,1.000,0
7,nominal_clean,planner_top1_platt,failure_proxy_h15,0.2,200,120,0.60,0.144094,0.000,-0.144094,0.000000,0.000,0.400,0
8,nominal_clean,planner_entropy_raw,failure_proxy_h15,0.2,200,0,0.00,0.999999,0.000,-0.999999,NaN,0.000,1.000,0
9,nominal_clean,planner_entropy_platt,failure_proxy_h15,0.2,200,200,1.00,0.138462,0.000,-0.138462,0.000000,0.000,0.000,0


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,label,shift_suite,n_rows,positive_rate,split_count_high_interaction_holdout,split_count_test
0,failure_proxy_h15,high_interaction_holdout,320,0.225,320.0,NaN
1,failure_proxy_h15,nominal_clean,200,0.000,NaN,200.0


[diagnostic] budget check by shift/variant


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,shift_suite,variant,empirical_failure_given_accepted,threshold,budget_violated
0,high_interaction_holdout,planner_top1_raw,NaN,0.2,0
1,high_interaction_holdout,planner_top1_platt,0.115385,0.2,0
2,high_interaction_holdout,planner_entropy_raw,NaN,0.2,0
3,high_interaction_holdout,planner_entropy_platt,0.225000,0.2,1
4,high_interaction_holdout,planner_combo_raw,NaN,0.2,0
5,high_interaction_holdout,planner_combo_platt,0.115385,0.2,0
6,nominal_clean,planner_top1_raw,NaN,0.2,0
7,nominal_clean,planner_top1_platt,0.000000,0.2,0
8,nominal_clean,planner_entropy_raw,NaN,0.2,0
9,nominal_clean,planner_entropy_platt,0.000000,0.2,0


## Step 9 - Export + Manifest (Completed)


In [10]:
if probe_bundle is None:
    stage_name = 'decision_audit_artifact_build_skipped'
    artifact_paths = {}
    metrics_path = None
    extra = {'run_skipped': 1}
else:
    stage_name = 'decision_audit_artifact_build_completed'
    artifact_paths = dict(probe_bundle.artifact_paths)
    metrics_path = str(artifact_paths.get('miscalibration_probe_summary', '')) or None
    extra = {
        'loaded_from_existing': int(bool(probe_bundle.loaded_from_existing)),
        'force_fresh_probe_build': int(bool(globals().get('FORCE_FRESH_PROBE_BUILD', False))),
        'auto_resume_after_restart': int(bool(globals().get('AUTO_RESUME_AFTER_RESTART', False))),
        'resume_sentinel_path': str(globals().get('SENTINEL_PATH', '')),
        'resume_saved_run_tag': str(globals().get('RUN_TAG', '')),
        'summary_rows': int(len(probe_bundle.benchmark_bundle.summary_df)),
        'per_shift_rows': int(len(probe_bundle.benchmark_bundle.per_shift_df)),
        'threshold_rows': int(len(probe_bundle.threshold_df)),
    }

manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name=NOTEBOOK_NAME,
    stage=stage_name,
    repo_dir=REPO_DIR,
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
    extra_fields=extra,
)

contract_paths = write_contract_storage_mirror(
    persist_root=PERSIST_ROOT,
    run_prefix=RUN_PREFIX,
    run_name=RUN_NAME,
    run_prefix_path=cfg.run_prefix,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    stage=stage_name,
    git_commit=str(globals().get('_RISK_UQ_REPO_REV', '')),
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    run_enabled=bool(RUN_ENABLED),
    artifact_paths=artifact_paths,
    metrics_csv_path=metrics_path,
    extra_fields=extra,
)
print('manifest_path =', manifest_path)
print('contract_run_manifest =', contract_paths['contract_run_manifest'])


manifest_path = /content/drive/MyDrive/waymax_experiments/risk_uq_runs/v2/risk_uq_v2_20260301_121046_notebook_contract_manifest.json
contract_run_manifest = /content/drive/MyDrive/waymax_experiments/risk_uq_runs/v2/risk_uq_v2_risk_uq_v2_20260301_121046/run_manifest.json
